# Bayesian Workflow: Linear Regression

This notebook applies a bounded Bayesian workflow to a synthetic linear-regression experiment. The model is small enough to inspect visually, yet its posterior requires numerical inference under the chosen priors.

## Question, estimands, and scope

**Question.** How does a continuous response change with a centered reference input, and what range of responses should the fitted model predict across the observed design?

- **Observation:** one pair $(x_i,y_i)$, where $x_i$ is a fixed, known reference input and $y_i$ is a continuous response.
- **Population:** comparable measurements generated across the declared input range under conditional independence, linearity, constant residual scale, and Gaussian errors.
- **Primary estimand:** $\beta$, the change in expected response for a one-unit increase in the centered input.
- **Secondary estimands:** $\alpha$, the expected response at $x=0$, and $\sigma$, the residual standard deviation.
- **Predictive target:** a future response at an input inside the observed range.

The data-generating process is

$$
x_i\in[-1.5,1.5],\qquad
\mu_i=\alpha_{\mathrm{true}}+\beta_{\mathrm{true}}x_i,\qquad
y_i\sim\operatorname{Normal}(\mu_i,\sigma_{\mathrm{true}}),
$$

with $n=16$, $\alpha_{\mathrm{true}}=1$, $\beta_{\mathrm{true}}=2$, and $\sigma_{\mathrm{true}}=0.75$. Because these data come from the fitted model family, the example tests reproducibility and synthetic self-consistency—not causal identification, real-sensor calibration, or adequacy for real data.

## Projection of the general Bayesian workflow

The repository's [Graphical Bayesian Workflow](../../.local-learning/Graphical_Bayesian_Workflow.md) is an iterative map, not a checklist or software architecture. This notebook follows the bounded path below; italic workflow labels in later sections identify each step.

```text
Question, estimands, and fixed design
  → Pick an initial linear-Gaussian model
  → Prior predictive check
      ├─ implausible → Modify the model or priors ↺
      └─ provisionally acceptable → Fit the model with NUTS
  → Validate computation
      ├─ invalid → Address computational issues and refit ↺
      └─ provisionally acceptable → Evaluate and use model
          └─ posterior summaries, joint update, and posterior predictive check
  → Human scientific review
      ├─ material problem → return to model or computation
      ├─ question unsupported → stop or narrow
      └─ adequate for this demonstration → persist and report
```

The likelihood, prior, and posterior plots make the Bayesian update inspectable; they are evidence for reasoning, not extra acceptance gates. Formal SBC, cross-validation, influence analysis, sensitivity analysis, model comparison, measurement-error models, nonlinear terms, and extrapolation remain outside this demonstration.

## Reproducible runtime and fixed settings

The first code cell checks the pinned runtime and exposes all data, prior, and sampling settings. Resolve any version mismatch before interpreting results.

In [ ]:
import json
import platform
import sys
from importlib.metadata import version
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import xarray as xr

EXPECTED_VERSIONS = {"pymc": "6.1.0", "arviz": "1.2.0", "pytensor": "3.1.3"}
package_versions = {
    package: version(package)
    for package in ("rag-pymc", "pymc", "arviz", "pytensor", "xarray", "h5netcdf")
}
version_mismatches = {
    package: {"expected": expected, "observed": package_versions[package]}
    for package, expected in EXPECTED_VERSIONS.items()
    if package_versions[package] != expected
}
if version_mismatches:
    raise RuntimeError(
        "Pinned runtime mismatch. Launch from the repository root with "
        "`uv run --extra notebooks jupyter lab examples/bayesian_workflow`. "
        f"Mismatches: {version_mismatches}"
    )

SEED = 20260726
N_OBSERVATIONS = 16
INTERCEPT_TRUE = 1.0
SLOPE_TRUE = 2.0
SIGMA_TRUE = 0.75
PREDICTOR_MIN = -1.5
PREDICTOR_MAX = 1.5
PRIOR_INTERCEPT_MEAN = 0.5
PRIOR_INTERCEPT_SD = 2.5
PRIOR_SLOPE_MEAN = 1.5
PRIOR_SLOPE_SD = 2.0
PRIOR_SIGMA_SD = 1.0
PRIOR_PREDICTIVE_DRAWS = 3000
INTERVAL_PROBABILITY = 0.94
INTERVAL_LOWER_QUANTILE = 0.03
INTERVAL_UPPER_QUANTILE = 0.97
if not np.isclose(INTERVAL_UPPER_QUANTILE - INTERVAL_LOWER_QUANTILE, INTERVAL_PROBABILITY):
    raise RuntimeError("The interval probability and quantiles are inconsistent.")
CHAINS = 4
DRAWS = 1000
TUNE = 1000
CORES = 1
CHAIN_SEEDS = [SEED + 10 + chain for chain in range(CHAINS)]

working_directory = Path.cwd()
if working_directory.name == "bayesian_workflow":
    example_directory = working_directory
elif (working_directory / "examples" / "bayesian_workflow").is_dir():
    example_directory = working_directory / "examples" / "bayesian_workflow"
else:
    raise RuntimeError("Launch Jupyter from the repository root or the example directory.")
output_directory = example_directory / "outputs" / "linear-regression-final"

runtime_and_settings = {
    "python_executable": sys.executable,
    "python_version": platform.python_version(),
    "package_versions": package_versions,
    "data_seed": SEED,
    "prior_predictive_seed": SEED + 1,
    "chain_seeds": CHAIN_SEEDS,
    "posterior_predictive_seed": SEED + 20,
    "interval_probability": INTERVAL_PROBABILITY,
    "sampling": {"chains": CHAINS, "draws": DRAWS, "tune": TUNE},
}
runtime_and_settings

## Deterministic constructed data

*Workflow: define the question and use **Fake data simulation** within **Validate computation**.*

The fixed predictor is evenly spaced and centered at zero. Thus $\alpha$ is the expected response at the design center, and intercept–slope posterior dependence is reduced. A local NumPy generator makes the data reproducible without changing global random state.

The observed pairs are

$$
\mathcal D=\{(x_i,y_i)\}_{i=1}^{16}.
$$

The reference inputs are treated as exact and fixed. Measurement error, selection, random $x$, and extrapolation beyond $[-1.5,1.5]$ are not modeled.

In [ ]:
predictor = np.linspace(PREDICTOR_MIN, PREDICTOR_MAX, N_OBSERVATIONS)
true_mean = INTERCEPT_TRUE + SLOPE_TRUE * predictor
data_rng = np.random.default_rng(SEED)
observations = data_rng.normal(loc=true_mean, scale=SIGMA_TRUE)

fig, axis = plt.subplots(figsize=(8, 4.5), layout="constrained")
axis.scatter(
    predictor,
    observations,
    color="#4C78A8",
    alpha=0.75,
    label=r"Observed responses $y$",
)
axis.plot(
    predictor,
    true_mean,
    color="#E45756",
    linewidth=2,
    label=r"Synthetic mean $\mu_{\mathrm{true}}(x)$",
)
axis.set(
    xlabel="Centered reference input x",
    ylabel="Continuous response y",
    title="Constructed linear-regression data",
)
axis.legend()
plt.show()

data_summary = {
    "shape": observations.shape,
    "predictor_mean": float(predictor.mean()),
    "predictor_range": [float(predictor.min()), float(predictor.max())],
    "response_mean": float(observations.mean()),
    "response_sd": float(observations.std(ddof=1)),
    "response_range": [float(observations.min()), float(observations.max())],
}
data_summary

## Generative model, priors, and graph checks

*Workflow: **Pick an initial model**; every component remains provisional.*

The fitted model is

$$
\begin{aligned}
\alpha &\sim \operatorname{Normal}(0.5,2.5),\\
\beta &\sim \operatorname{Normal}(1.5,2),\\
\sigma &\sim \operatorname{HalfNormal}(1),\\
\mu_i &= \alpha+\beta x_i,\\
y_i\mid\alpha,\beta,\sigma,x_i &\sim \operatorname{Normal}(\mu_i,\sigma).
\end{aligned}
$$

These are synthetic regularizing assumptions, not domain-elicited priors. They center the intercept at $0.5$, the slope at $1.5$ response units per input unit, and constrain $\sigma>0$. Their implications are assessed in observable space by prior predictive simulation.

`coords` and `dims` name the observation axis. `mu` remains a symbolic `pm.Deterministic`, so each posterior draw contains an entire regression line over the fixed design.

In [ ]:
coords = {"observation": np.arange(N_OBSERVATIONS, dtype=np.int64)}
with pm.Model(coords=coords) as model:
    intercept = pm.Normal("intercept", mu=PRIOR_INTERCEPT_MEAN, sigma=PRIOR_INTERCEPT_SD)
    slope = pm.Normal("slope", mu=PRIOR_SLOPE_MEAN, sigma=PRIOR_SLOPE_SD)
    sigma = pm.HalfNormal("sigma", sigma=PRIOR_SIGMA_SD)
    mu = pm.Deterministic("mu", intercept + slope * predictor, dims="observation")
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=observations, dims="observation")

initial_log_probability = float(model.compile_logp()(model.initial_point()))
implementation_checks = {
    "free_variables": [variable.name for variable in model.free_RVs],
    "deterministic_variables": [variable.name for variable in model.deterministics],
    "observed_variables": [variable.name for variable in model.observed_RVs],
    "mu_dimension": model.named_vars_to_dims["mu"],
    "y_dimension": model.named_vars_to_dims["y"],
    "observation_coordinate_size": len(model.coords["observation"]),
    "initial_log_probability": initial_log_probability,
    "initial_log_probability_is_finite": bool(np.isfinite(initial_log_probability)),
}
implementation_checks

## Joint likelihood in three-dimensional parameter space

*Workflow: inspect the likelihood component of **Pick an initial model** before fitting.*

Holding $\mathbf{x}$ and the complete observed $\mathbf{y}$ fixed makes the Gaussian likelihood a function of the unknown parameters:

$$
L(\alpha,\beta,\sigma;\mathbf{y},\mathbf{x})
=\prod_{i=1}^{n}\operatorname{Normal}
\left(y_i\mid\alpha+\beta x_i,\sigma\right),
$$

or, on the numerically stable log scale,

$$
\ell(\alpha,\beta,\sigma)
=-n\log\sigma-\frac{1}{2\sigma^2}
\sum_{i=1}^{n}(y_i-\alpha-\beta x_i)^2+C.
$$

The finite grid is centered on the Gaussian maximum-likelihood estimate, computed without the priors. Here $\hat\sigma_{\mathrm{MLE}}=\sqrt{\mathrm{RSS}/n}$; the later OLS check uses $\sqrt{\mathrm{RSS}/(n-2)}$, so the scale estimates differ by definition. Grid bounds aid visualization and are not confidence or credible intervals.

Each plotted point is one combination $(\alpha,\beta,\sigma)$. Its color is mapped linearly to the relative likelihood

$$
L_{\mathrm{rel}}=\frac{L}{L_{\max}}=\exp(\ell-\max\ell),
$$

Color and opacity encode $L_{\mathrm{rel}}$; a declared cutoff hides near-zero values for legibility. The black cross is the MLE and the red star is the synthetic truth. This is a finite-grid likelihood view—not probability over parameters or a posterior.

In [ ]:
centered_predictor = predictor - predictor.mean()
centered_observations = observations - observations.mean()
predictor_sum_of_squares = float(np.sum(centered_predictor**2))
likelihood_slope_mle = float(
    np.dot(centered_predictor, centered_observations) / predictor_sum_of_squares
)
likelihood_intercept_mle = float(observations.mean() - likelihood_slope_mle * predictor.mean())
likelihood_mle_residuals = observations - (
    likelihood_intercept_mle + likelihood_slope_mle * predictor
)
likelihood_sigma_mle = float(np.sqrt(np.mean(likelihood_mle_residuals**2)))

intercept_local_scale = likelihood_sigma_mle * np.sqrt(
    1 / N_OBSERVATIONS + predictor.mean() ** 2 / predictor_sum_of_squares
)
slope_local_scale = likelihood_sigma_mle / np.sqrt(predictor_sum_of_squares)
intercept_grid = np.linspace(
    likelihood_intercept_mle - 6 * intercept_local_scale,
    likelihood_intercept_mle + 6 * intercept_local_scale,
    41,
)
slope_grid = np.linspace(
    likelihood_slope_mle - 6 * slope_local_scale,
    likelihood_slope_mle + 6 * slope_local_scale,
    41,
)
sigma_grid = likelihood_sigma_mle * np.linspace(0.25, 3.0, 56)

intercept_surface, slope_surface = np.meshgrid(intercept_grid, slope_grid, indexing="ij")
mean_surface = intercept_surface[..., None] + slope_surface[..., None] * predictor
residual_sum_of_squares = np.sum((observations - mean_surface) ** 2, axis=-1)
sigma_broadcast = sigma_grid[None, None, :]
joint_log_likelihood = (
    -N_OBSERVATIONS * np.log(sigma_broadcast)
    - 0.5 * N_OBSERVATIONS * np.log(2 * np.pi)
    - residual_sum_of_squares[..., None] / (2 * sigma_broadcast**2)
)
maximum_log_likelihood = float(joint_log_likelihood.max())
relative_likelihood = np.exp(joint_log_likelihood - maximum_log_likelihood)

intercept_volume, slope_volume, sigma_volume = np.meshgrid(
    intercept_grid, slope_grid, sigma_grid, indexing="ij"
)
grid_maximum_index = np.unravel_index(np.argmax(joint_log_likelihood), joint_log_likelihood.shape)
grid_maximum = {
    "intercept": float(intercept_volume[grid_maximum_index]),
    "slope": float(slope_volume[grid_maximum_index]),
    "sigma": float(sigma_volume[grid_maximum_index]),
}
likelihood_mle = {
    "intercept": likelihood_intercept_mle,
    "slope": likelihood_slope_mle,
    "sigma": likelihood_sigma_mle,
}
grid_steps = {
    "intercept": float(intercept_grid[1] - intercept_grid[0]),
    "slope": float(slope_grid[1] - slope_grid[0]),
    "sigma": float(sigma_grid[1] - sigma_grid[0]),
}
for parameter in ("intercept", "slope", "sigma"):
    assert abs(grid_maximum[parameter] - likelihood_mle[parameter]) <= (
        grid_steps[parameter] / 2 + np.finfo(float).eps
    )

LIKELIHOOD_VISIBILITY_CUTOFF = 1e-3
visible = relative_likelihood >= LIKELIHOOD_VISIBILITY_CUTOFF
visible_relative_likelihood = relative_likelihood[visible]
render_order = np.argsort(visible_relative_likelihood)
likelihood_colormap = plt.get_cmap("viridis")
point_colors = likelihood_colormap(visible_relative_likelihood)
point_colors[:, 3] = 0.08 + 0.82 * visible_relative_likelihood

fig = plt.figure(figsize=(10, 7), layout="constrained")
axis = fig.add_subplot(111, projection="3d", computed_zorder=False)
axis.scatter(
    intercept_volume[visible][render_order],
    slope_volume[visible][render_order],
    sigma_volume[visible][render_order],
    c=point_colors[render_order],
    s=16,
    edgecolors="none",
    depthshade=False,
    zorder=1,
)
axis.scatter(
    likelihood_intercept_mle,
    likelihood_slope_mle,
    likelihood_sigma_mle,
    color="black",
    edgecolor="white",
    marker="X",
    s=220,
    linewidth=1.4,
    depthshade=False,
    label="Gaussian MLE",
    zorder=5,
)
axis.scatter(
    INTERCEPT_TRUE,
    SLOPE_TRUE,
    SIGMA_TRUE,
    color="#E45756",
    edgecolor="black",
    marker="*",
    s=280,
    linewidth=1.2,
    depthshade=False,
    label="Synthetic generating value",
    zorder=5,
)
axis.set(
    xlabel=r"Intercept $\alpha$",
    ylabel=r"Slope $\beta$",
    zlabel=r"Residual scale $\sigma$",
    title=(
        r"3D likelihood heatmap for fixed $\mathbf{x}$ and $\mathbf{y}$"
        + "\n"
        + rf"Visible region: $L/L_{{\max}} \geq {LIKELIHOOD_VISIBILITY_CUTOFF:g}$"
    ),
)
axis.view_init(elev=24, azim=-58)
axis.set_box_aspect((1.1, 1.1, 0.9))
axis.legend(loc="upper left")
colorbar_mappable = plt.cm.ScalarMappable(cmap=likelihood_colormap)
colorbar_mappable.set_clim(0, 1)
colorbar_mappable.set_array(np.array([0.0, 1.0]))
colorbar = fig.colorbar(colorbar_mappable, ax=axis, shrink=0.68, pad=0.1)
colorbar.set_label(r"Relative likelihood $L/L_{\max}$")
plt.show()

likelihood_volume_summary = {
    "fixed_dataset_size": N_OBSERVATIONS,
    "grid_shape": relative_likelihood.shape,
    "grid_points": int(relative_likelihood.size),
    "visibility_cutoff": LIKELIHOOD_VISIBILITY_CUTOFF,
    "visible_points": int(visible.sum()),
    "mle": likelihood_mle,
    "grid_maximum": grid_maximum,
}
likelihood_volume_summary

## Prior predictive check

*Workflow: **Prior predictive check**; implausible implications route to **Modify the model**.*

Before conditioning on observations, the prior predictive distribution is

$$
p(y^{\mathrm{rep}}\mid x)=\int p(y^{\mathrm{rep}}\mid x,\alpha,\beta,\sigma)
p(\alpha)p(\beta)p(\sigma)\,d\alpha\,d\beta\,d\sigma.
$$

A prior-predictive outcome $y^{\mathrm{rep}}$ is a hypothetical response generated before observing $\mathbf{y}$:

$$
(\alpha^{(s)},\beta^{(s)},\sigma^{(s)})\sim p(\alpha,\beta,\sigma),\qquad
\mu^{(s)}(x)=\alpha^{(s)}+\beta^{(s)}x,\qquad
y^{\mathrm{rep},(s)}(x)\sim\operatorname{Normal}(\mu^{(s)}(x),\sigma^{(s)}).
$$

These simulations ask which response values and regression patterns the complete model permits before seeing $\mathbf{y}$. Both plots retain the relationship with $x$.

The pointwise figure separates:

- the prior median and 94% interval for the conditional mean $\mu(x)$;
- the wider 94% interval for $y^{\mathrm{rep}}$, which also includes residual variation $\sigma$.

A real analysis would set plausible observable ranges from domain knowledge before seeing outcomes. Here the check assesses only coherence with the declared synthetic scale.

In [ ]:
with model:
    prior_predictive = pm.sample_prior_predictive(
        draws=PRIOR_PREDICTIVE_DRAWS,
        var_names=["intercept", "slope", "sigma", "mu", "y"],
        random_seed=SEED + 1,
    )
assert isinstance(prior_predictive, xr.DataTree)

prior_mu_samples = prior_predictive["prior"].dataset["mu"]
prior_mu = prior_mu_samples.values.reshape(-1, N_OBSERVATIONS)
prior_sigma = prior_predictive["prior"].dataset["sigma"]
prior_y = prior_predictive["prior_predictive"].dataset["y"]
prior_predictive_summary = {
    "interval_probability": INTERVAL_PROBABILITY,
    "sigma_median": float(prior_sigma.median().item()),
    "sigma_q95": float(prior_sigma.quantile(0.95).item()),
}

fig, axis = plt.subplots(figsize=(9, 5), layout="constrained")
for prior_line in prior_mu[:60]:
    axis.plot(predictor, prior_line, color="#4C78A8", alpha=0.12)
axis.set(
    xlabel="Centered reference input x",
    ylabel=r"Conditional mean $\mu(x)$",
    title=r"Prior draws of the mean function $\mu(x)$",
)
plt.show()

interval_quantiles = [INTERVAL_LOWER_QUANTILE, 0.5, INTERVAL_UPPER_QUANTILE]
prior_mu_quantiles = prior_mu_samples.quantile(interval_quantiles, dim=("chain", "draw"))
prior_predictive_quantiles = prior_y.quantile(interval_quantiles, dim=("chain", "draw"))
fig, axis = plt.subplots(figsize=(9, 5), layout="constrained")
axis.fill_between(
    predictor,
    prior_predictive_quantiles.sel(quantile=INTERVAL_LOWER_QUANTILE),
    prior_predictive_quantiles.sel(quantile=INTERVAL_UPPER_QUANTILE),
    color="#72B7B2",
    alpha=0.22,
    label=r"Pointwise 94% prior predictive interval for $y^{\mathrm{rep}}$",
)
axis.fill_between(
    predictor,
    prior_mu_quantiles.sel(quantile=INTERVAL_LOWER_QUANTILE),
    prior_mu_quantiles.sel(quantile=INTERVAL_UPPER_QUANTILE),
    color="#F58518",
    alpha=0.3,
    label=r"Pointwise 94% prior interval for mean $\mu(x)$",
)
axis.plot(
    predictor,
    prior_mu_quantiles.sel(quantile=0.5),
    color="#A14B00",
    linewidth=2,
    label=r"Prior median of the conditional mean $\mu(x)$",
)
axis.set(
    xlabel="Centered reference input x",
    ylabel="Continuous response y",
    title="Prior predictive implications before observing y",
)
axis.legend(loc="upper left")
plt.show()
prior_predictive_summary

## Joint prior distribution in parameter space

*Workflow: inspect the joint prior within **Pick an initial model**, before conditioning on data.*

The three prior components are independent by construction, so their joint density factorizes as

$$
p(\alpha,\beta,\sigma)
=p(\alpha)\,p(\beta)\,p(\sigma),
$$

Relative to its boundary mode $(0.5,1.5,0)$, the joint density is

$$
\frac{p(\alpha,\beta,\sigma)}{p(0.5,1.5,0)}
=\exp\!\left[-\frac{1}{2}\left(
\frac{(\alpha-0.5)^2}{2.5^2}+
\frac{(\beta-1.5)^2}{2^2}+
\frac{\sigma^2}{1^2}
\right)\right],\qquad \sigma\geq0.
$$

Although $\sigma$ is half-normal, its kernel on $\sigma\geq0$ is Gaussian; the truncation constant cancels in this density ratio. Each point is a PyMC joint-prior draw. Position shows the empirical draw, color and opacity show its analytical relative density, and no observed $y$ enters. The black cross marks the prior mode; the red star is synthetic truth shown only for reference.

In [ ]:
prior_parameter_dataset = prior_predictive["prior"].dataset
prior_intercept_draws = prior_parameter_dataset["intercept"].values.ravel()
prior_slope_draws = prior_parameter_dataset["slope"].values.ravel()
prior_sigma_draws = prior_parameter_dataset["sigma"].values.ravel()
prior_parameter_draws = np.column_stack(
    [prior_intercept_draws, prior_slope_draws, prior_sigma_draws]
)
assert prior_parameter_draws.shape == (PRIOR_PREDICTIVE_DRAWS, 3)
assert np.isfinite(prior_parameter_draws).all()
assert (prior_sigma_draws >= 0).all()

prior_standardized_squared_radius = (
    ((prior_intercept_draws - PRIOR_INTERCEPT_MEAN) / PRIOR_INTERCEPT_SD) ** 2
    + ((prior_slope_draws - PRIOR_SLOPE_MEAN) / PRIOR_SLOPE_SD) ** 2
    + (prior_sigma_draws / PRIOR_SIGMA_SD) ** 2
)
prior_relative_joint_density = np.exp(-0.5 * prior_standardized_squared_radius)
assert ((prior_relative_joint_density > 0) & (prior_relative_joint_density <= 1)).all()
prior_render_order = np.argsort(prior_relative_joint_density)
prior_colormap = plt.get_cmap("viridis")
prior_point_colors = prior_colormap(prior_relative_joint_density)
prior_point_colors[:, 3] = 0.12 + 0.78 * prior_relative_joint_density

prior_intercept_bounds = (
    min(
        float(prior_intercept_draws.min()),
        PRIOR_INTERCEPT_MEAN - 3.5 * PRIOR_INTERCEPT_SD,
    ),
    max(
        float(prior_intercept_draws.max()),
        PRIOR_INTERCEPT_MEAN + 3.5 * PRIOR_INTERCEPT_SD,
    ),
)
prior_slope_bounds = (
    min(
        float(prior_slope_draws.min()),
        PRIOR_SLOPE_MEAN - 3.5 * PRIOR_SLOPE_SD,
    ),
    max(
        float(prior_slope_draws.max()),
        PRIOR_SLOPE_MEAN + 3.5 * PRIOR_SLOPE_SD,
    ),
)
prior_sigma_upper_bound = max(float(prior_sigma_draws.max()), 3.5 * PRIOR_SIGMA_SD)
prior_intercept_padding = 0.03 * (prior_intercept_bounds[1] - prior_intercept_bounds[0])
prior_slope_padding = 0.03 * (prior_slope_bounds[1] - prior_slope_bounds[0])
prior_joint_axis_limits = {
    "intercept": (
        prior_intercept_bounds[0] - prior_intercept_padding,
        prior_intercept_bounds[1] + prior_intercept_padding,
    ),
    "slope": (
        prior_slope_bounds[0] - prior_slope_padding,
        prior_slope_bounds[1] + prior_slope_padding,
    ),
    "sigma": (0.0, 1.03 * prior_sigma_upper_bound),
}

fig = plt.figure(figsize=(10, 7), layout="constrained")
axis = fig.add_subplot(111, projection="3d", computed_zorder=False)
axis.scatter(
    prior_intercept_draws[prior_render_order],
    prior_slope_draws[prior_render_order],
    prior_sigma_draws[prior_render_order],
    c=prior_point_colors[prior_render_order],
    s=24,
    edgecolors="none",
    depthshade=False,
    zorder=1,
)
axis.scatter(
    PRIOR_INTERCEPT_MEAN,
    PRIOR_SLOPE_MEAN,
    0.0,
    color="black",
    edgecolor="white",
    marker="X",
    s=220,
    linewidth=1.4,
    depthshade=False,
    label="Joint prior density mode",
    zorder=5,
)
axis.scatter(
    INTERCEPT_TRUE,
    SLOPE_TRUE,
    SIGMA_TRUE,
    color="#E45756",
    edgecolor="black",
    marker="*",
    s=280,
    linewidth=1.2,
    depthshade=False,
    label="Synthetic generating value",
    zorder=5,
)
axis.set(
    xlabel=r"Intercept $\alpha$",
    ylabel=r"Slope $\beta$",
    zlabel=r"Residual scale $\sigma$",
    title=(
        r"Joint prior draws for $(\alpha,\beta,\sigma)$"
        + "\n"
        + "Position: PyMC draw; color: analytical relative prior density"
    ),
)
axis.set_xlim(prior_joint_axis_limits["intercept"])
axis.set_ylim(prior_joint_axis_limits["slope"])
axis.set_zlim(prior_joint_axis_limits["sigma"])
axis.view_init(elev=24, azim=-58)
axis.set_box_aspect((1.1, 1.1, 0.9))
axis.legend(loc="upper left")
prior_colorbar_mappable = plt.cm.ScalarMappable(cmap=prior_colormap)
prior_colorbar_mappable.set_clim(0, 1)
prior_colorbar_mappable.set_array(np.array([0.0, 1.0]))
prior_colorbar = fig.colorbar(prior_colorbar_mappable, ax=axis, shrink=0.68, pad=0.1)
prior_colorbar.set_label(r"Relative joint prior density $p(\theta)/p_{\max}$")
plt.show()

prior_joint_summary = {
    "draws": int(prior_parameter_draws.shape[0]),
    "source": "PyMC prior group",
    "factorization": "p(intercept) * p(slope) * p(sigma)",
    "relative_density_range": [
        float(prior_relative_joint_density.min()),
        float(prior_relative_joint_density.max()),
    ],
    "axis_limits": prior_joint_axis_limits,
}
prior_joint_summary

## Constructed-data and independent least-squares check

*Workflow: **Fake data simulation** and implementation checks within **Validate computation**.*

Ordinary least squares independently calculates the line that minimizes squared residuals:

$$
\hat{\boldsymbol\beta}_{\mathrm{OLS}}=(X^\top X)^{-1}X^\top y,
\qquad X=[\mathbf 1,\mathbf x].
$$

`numpy.linalg.lstsq` avoids explicitly forming the inverse. OLS is not a posterior oracle: it omits the priors and uncertainty, but independently checks the fitted line's direction and scale.

In [ ]:
design_matrix = np.column_stack([np.ones(N_OBSERVATIONS), predictor])
ols_coefficients, _, _, _ = np.linalg.lstsq(design_matrix, observations, rcond=None)
ols_fitted = design_matrix @ ols_coefficients
ols_residuals = observations - ols_fitted
ols_residual_sd = float(
    np.sqrt(np.sum(ols_residuals**2) / (N_OBSERVATIONS - design_matrix.shape[1]))
)
ols_summary = {
    "intercept": float(ols_coefficients[0]),
    "slope": float(ols_coefficients[1]),
    "residual_sd": ols_residual_sd,
}
ols_summary

## Posterior and posterior predictive sampling

*Workflow: **Fit the model**.*

Bayes' rule combines the likelihood and priors:

$$
p(\alpha,\beta,\sigma\mid y,x)\propto
\left[\prod_{i=1}^{n}\operatorname{Normal}(y_i\mid\alpha+\beta x_i,\sigma)\right]
p(\alpha)p(\beta)p(\sigma).
$$

No closed-form posterior is available under these priors. `pm.sample` uses NUTS with four independently seeded chains; `pm.sample_posterior_predictive` then simulates response vectors at the fixed design and appends them to the `xarray.DataTree`.

In [ ]:
with model:
    posterior = pm.sample(
        chains=CHAINS,
        draws=DRAWS,
        tune=TUNE,
        random_seed=CHAIN_SEEDS,
        cores=CORES,
        progressbar=True,
    )
    posterior = pm.sample_posterior_predictive(
        posterior,
        var_names=["y"],
        random_seed=SEED + 20,
        progressbar=True,
        extend_inferencedata=True,
    )
assert isinstance(posterior, xr.DataTree)
posterior.groups

## Computational diagnostics

*Workflow: **Validate computation**; a material issue routes to **Addressing computational issues** and refitting.*

We inspect complementary evidence for `intercept`, `slope`, and `sigma`:

- rank-normalized split $\widehat R$ for between/within-chain compatibility;
- bulk and tail effective sample sizes for central and tail summaries;
- MCSE for posterior means;
- divergences, maximum observed tree depth, and per-chain BFMI;
- trace and rank plots for visible mixing or chain-specific behavior.

Together these assess numerical reliability for this run. They do not assess the scientific adequacy of the likelihood, priors, estimands, or question.

In [ ]:
PARAMETER_NAMES = ["intercept", "slope", "sigma"]
r_hat_result = az.rhat(posterior, var_names=PARAMETER_NAMES, method="rank")
ess_bulk_result = az.ess(posterior, var_names=PARAMETER_NAMES, method="bulk")
ess_tail_result = az.ess(posterior, var_names=PARAMETER_NAMES, method="tail")
mcse_result = az.mcse(posterior, var_names=PARAMETER_NAMES, method="mean")
diagnostics_by_parameter = {
    parameter: {
        "r_hat_rank": float(r_hat_result[parameter].item()),
        "ess_bulk": float(ess_bulk_result[parameter].item()),
        "ess_tail": float(ess_tail_result[parameter].item()),
        "mcse_mean": float(mcse_result[parameter].item()),
    }
    for parameter in PARAMETER_NAMES
}
sample_stats = posterior["sample_stats"].dataset
divergences_by_chain = sample_stats["diverging"].sum(dim="draw").astype(int).values.tolist()
bfmi_result = az.bfmi(posterior)
bfmi_by_chain = bfmi_result.dataset["energy"].values.tolist()
diagnostics_overview = {
    "divergences_total": int(sample_stats["diverging"].sum().item()),
    "divergences_by_chain": divergences_by_chain,
    "maximum_observed_tree_depth": int(sample_stats["tree_depth"].max().item()),
    "bfmi_by_chain": [float(value) for value in bfmi_by_chain],
}
display(
    az.summary(
        posterior,
        var_names=PARAMETER_NAMES,
        kind="all",
        ci_prob=INTERVAL_PROBABILITY,
        ci_kind="eti",
    )
)
display(diagnostics_by_parameter)
display(diagnostics_overview)

In [ ]:
az.plot_trace(posterior, var_names=PARAMETER_NAMES)
plt.show()
az.plot_rank(posterior, var_names=PARAMETER_NAMES)
plt.show()

## Bayesian updating: prior versus posterior

*Workflow: inspect **Influence of prior** within **Evaluate and use model**.*

Bayes' rule updates the joint distribution; each displayed parameter posterior is a marginal:

$$
p(\theta_j\mid x,y)=\int p(\theta\mid x,y)\,d\theta_{-j}.
$$

Each panel uses one parameter scale and shared histogram bins, making changes in location and uncertainty visible. These are parameter—not predictive—distributions. Synthetic truth is known only because the data are constructed.

In [ ]:
posterior_dataset = posterior["posterior"].dataset
parameter_plot_metadata = {
    "intercept": (r"\alpha", r"Intercept $\alpha$", INTERCEPT_TRUE),
    "slope": (r"\beta", r"Slope $\beta$", SLOPE_TRUE),
    "sigma": (r"\sigma", r"Residual scale $\sigma$", SIGMA_TRUE),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), layout="constrained")
for axis, (parameter, (symbol, title, truth)) in zip(
    axes, parameter_plot_metadata.items(), strict=True
):
    prior_values = prior_predictive["prior"].dataset[parameter].values.ravel()
    posterior_values = posterior_dataset[parameter].values.ravel()
    lower = min(float(prior_values.min()), float(posterior_values.min()))
    upper = max(float(prior_values.max()), float(posterior_values.max()))
    padding = 0.02 * (upper - lower)
    lower = 0.0 if parameter == "sigma" else lower - padding
    upper += padding
    shared_bins = np.linspace(lower, upper, 55)
    axis.hist(
        prior_values,
        bins=shared_bins,
        density=True,
        histtype="step",
        linewidth=2,
        color="#F58518",
        label=rf"Prior $p({symbol})$",
    )
    axis.hist(
        posterior_values,
        bins=shared_bins,
        density=True,
        alpha=0.45,
        color="#4C78A8",
        label=rf"Posterior $p({symbol}\mid x,y)$",
    )
    axis.axvline(
        truth,
        color="#E45756",
        linestyle="--",
        linewidth=2,
        label="Synthetic truth",
    )
    axis.set(xlabel="Parameter value", ylabel="Density", title=title)
    axis.legend()

fig.suptitle("Bayesian updating: prior and posterior parameter distributions", fontsize=14)
plt.show()

## Joint posterior geometry

*Workflow: understand posterior geometry after **Validate computation**.* Marginal plots cannot reveal dependence among parameters.

The inferential object is the joint distribution

$$
p(\alpha,\beta,\sigma\mid x,y),
$$

not three unrelated marginals. The figure shows all pairwise projections of the same retained draws:

- $(\alpha,\beta)$: intercept–slope trade-offs;
- $(\alpha,\sigma)$: baseline–residual-scale trade-offs;
- $(\beta,\sigma)$: slope–residual-scale trade-offs.

Because $\bar x=0$, $\alpha$ is the mean at the design center and the intercept and slope columns are orthogonal. Weak linear dependence between $\alpha$ and $\beta$ is therefore plausible here, not guaranteed in every centered model.

Each point is one draw in the original parameterization. Pearson correlation measures only linear dependence: a value near zero neither proves independence or convergence nor excludes nonlinear or multimodal geometry.

In [ ]:
joint_parameter_metadata = {
    "intercept": {"label": r"Intercept $\alpha$", "truth": INTERCEPT_TRUE},
    "slope": {"label": r"Slope $\beta$", "truth": SLOPE_TRUE},
    "sigma": {"label": r"Residual scale $\sigma$", "truth": SIGMA_TRUE},
}
joint_parameter_pairs = [
    ("intercept", "slope"),
    ("intercept", "sigma"),
    ("slope", "sigma"),
]
joint_draw_matrix = np.column_stack(
    [posterior_dataset[parameter].values.ravel() for parameter in PARAMETER_NAMES]
)
joint_parameter_index = {parameter: index for index, parameter in enumerate(PARAMETER_NAMES)}
joint_correlation_matrix = np.corrcoef(joint_draw_matrix, rowvar=False)
if not (
    np.all(np.isfinite(joint_correlation_matrix))
    and np.allclose(joint_correlation_matrix, joint_correlation_matrix.T)
    and np.allclose(np.diag(joint_correlation_matrix), 1.0)
):
    raise RuntimeError("The joint posterior correlation matrix is invalid.")

pairwise_correlations = {
    f"{first}_{second}": float(
        joint_correlation_matrix[joint_parameter_index[first], joint_parameter_index[second]]
    )
    for first, second in joint_parameter_pairs
}
joint_geometry_summary = {
    "parameter_order": PARAMETER_NAMES,
    "posterior_draws_visualized": int(joint_draw_matrix.shape[0]),
    "pearson_correlation_matrix": joint_correlation_matrix.tolist(),
    "pairwise_pearson_correlations": pairwise_correlations,
}

panel_colors = ["#4C78A8", "#F58518", "#54A24B"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), layout="constrained")
for axis, (first, second), color in zip(axes, joint_parameter_pairs, panel_colors, strict=True):
    first_values = joint_draw_matrix[:, joint_parameter_index[first]]
    second_values = joint_draw_matrix[:, joint_parameter_index[second]]
    correlation = pairwise_correlations[f"{first}_{second}"]
    axis.scatter(
        first_values,
        second_values,
        s=11,
        alpha=0.18,
        color=color,
        rasterized=True,
        label="Posterior draws",
    )
    axis.scatter(
        joint_parameter_metadata[first]["truth"],
        joint_parameter_metadata[second]["truth"],
        marker="X",
        s=120,
        color="#E45756",
        label="Synthetic truth",
    )
    axis.set(
        xlabel=joint_parameter_metadata[first]["label"],
        ylabel=joint_parameter_metadata[second]["label"],
        title=(f"{first.capitalize()} and {second}\nPearson correlation = {correlation:.3f}"),
    )
    axis.legend()

fig.suptitle("Joint posterior geometry of intercept, slope, and residual scale", fontsize=14)
plt.show()
joint_geometry_summary

## Bayes' rule in joint parameter space: prior × likelihood → posterior

*Workflow: connect **Pick an initial model**, **Fit the model**, and posterior understanding after **Validate computation**.*

Let $\theta=(\alpha,\beta,\sigma)$. For the fixed design $\mathbf{x}$ and observed dataset $\mathbf{y}$, Bayes' rule is

$$
p(\theta\mid\mathbf{x},\mathbf{y})
=\frac{p(\theta)L(\theta;\mathbf{x},\mathbf{y})}
{\int p(\theta')L(\theta';\mathbf{x},\mathbf{y})\,d\theta'}
\propto p(\theta)L(\theta;\mathbf{x},\mathbf{y}).
$$

The panels use one camera angle and parameter order:

1. **Prior:** joint PyMC draws on the full prior scale; the dashed box is only the zoom window used later.
2. **Likelihood:** relative likelihood for fixed $\mathbf{x},\mathbf{y}$; it is not probability over $\theta$.
3. **Posterior:** MCMC draws on the likelihood limits, colored by the relative kernel $p(\theta)L(\theta)$.

The broader prior needs its own axes, while likelihood and posterior share limits. The code verifies that this fixed window contains every retained posterior draw rather than silently clipping them. Colors are normalized separately, so equal colors across panels are not equal densities. The figure illustrates the conditional update; it does not prove sampling correctness or model adequacy.

In [ ]:
comparison_axis_limits = {
    "intercept": (float(intercept_grid[0]), float(intercept_grid[-1])),
    "slope": (float(slope_grid[0]), float(slope_grid[-1])),
    "sigma": (float(sigma_grid[0]), float(sigma_grid[-1])),
}

intercept_low, intercept_high = comparison_axis_limits["intercept"]
slope_low, slope_high = comparison_axis_limits["slope"]
sigma_low, sigma_high = comparison_axis_limits["sigma"]
comparison_box_corners = np.array(
    [
        [intercept_low, slope_low, sigma_low],
        [intercept_high, slope_low, sigma_low],
        [intercept_low, slope_high, sigma_low],
        [intercept_high, slope_high, sigma_low],
        [intercept_low, slope_low, sigma_high],
        [intercept_high, slope_low, sigma_high],
        [intercept_low, slope_high, sigma_high],
        [intercept_high, slope_high, sigma_high],
    ]
)
comparison_box_edges = [
    (0, 1),
    (0, 2),
    (1, 3),
    (2, 3),
    (4, 5),
    (4, 6),
    (5, 7),
    (6, 7),
    (0, 4),
    (1, 5),
    (2, 6),
    (3, 7),
]

posterior_intercept_draws = joint_draw_matrix[:, 0]
posterior_slope_draws = joint_draw_matrix[:, 1]
posterior_sigma_draws = joint_draw_matrix[:, 2]
posterior_log_prior = -0.5 * (
    ((posterior_intercept_draws - PRIOR_INTERCEPT_MEAN) / PRIOR_INTERCEPT_SD) ** 2
    + ((posterior_slope_draws - PRIOR_SLOPE_MEAN) / PRIOR_SLOPE_SD) ** 2
    + (posterior_sigma_draws / PRIOR_SIGMA_SD) ** 2
)
posterior_mean_draws = (
    posterior_intercept_draws[:, None] + posterior_slope_draws[:, None] * predictor[None, :]
)
posterior_residual_sum_of_squares = np.sum(
    (observations[None, :] - posterior_mean_draws) ** 2, axis=1
)
posterior_log_likelihood = (
    -N_OBSERVATIONS * np.log(posterior_sigma_draws)
    - 0.5 * N_OBSERVATIONS * np.log(2 * np.pi)
    - posterior_residual_sum_of_squares / (2 * posterior_sigma_draws**2)
)
posterior_log_kernel = posterior_log_prior + posterior_log_likelihood
posterior_relative_kernel = np.exp(posterior_log_kernel - posterior_log_kernel.max())
if not (
    np.isfinite(posterior_relative_kernel).all()
    and ((posterior_relative_kernel > 0) & (posterior_relative_kernel <= 1)).all()
):
    raise RuntimeError("The relative posterior kernel is invalid.")
posterior_comparison_order = np.argsort(posterior_relative_kernel)

prior_inside_comparison_window = (
    (prior_intercept_draws >= intercept_low)
    & (prior_intercept_draws <= intercept_high)
    & (prior_slope_draws >= slope_low)
    & (prior_slope_draws <= slope_high)
    & (prior_sigma_draws >= sigma_low)
    & (prior_sigma_draws <= sigma_high)
)
posterior_inside_comparison_window = (
    (posterior_intercept_draws >= intercept_low)
    & (posterior_intercept_draws <= intercept_high)
    & (posterior_slope_draws >= slope_low)
    & (posterior_slope_draws <= slope_high)
    & (posterior_sigma_draws >= sigma_low)
    & (posterior_sigma_draws <= sigma_high)
)
if not posterior_inside_comparison_window.all():
    raise RuntimeError("The shared likelihood/posterior window clips retained posterior draws.")

comparison_colormap = plt.get_cmap("viridis")
posterior_comparison_colors = comparison_colormap(posterior_relative_kernel)
posterior_comparison_colors[:, 3] = 0.10 + 0.80 * posterior_relative_kernel

fig = plt.figure(figsize=(21, 7.2), layout="constrained")
prior_axis = fig.add_subplot(131, projection="3d", computed_zorder=False)
likelihood_axis = fig.add_subplot(132, projection="3d", computed_zorder=False)
posterior_axis = fig.add_subplot(133, projection="3d", computed_zorder=False)

prior_axis.scatter(
    prior_intercept_draws[prior_render_order],
    prior_slope_draws[prior_render_order],
    prior_sigma_draws[prior_render_order],
    c=prior_point_colors[prior_render_order],
    s=13,
    edgecolors="none",
    depthshade=False,
    zorder=1,
)
for edge_index, (start, end) in enumerate(comparison_box_edges):
    edge = comparison_box_corners[[start, end]]
    prior_axis.plot(
        edge[:, 0],
        edge[:, 1],
        edge[:, 2],
        color="black",
        linestyle="--",
        linewidth=1.4,
        label="Likelihood/posterior zoom window" if edge_index == 0 else None,
        zorder=4,
    )
prior_axis.scatter(
    INTERCEPT_TRUE,
    SLOPE_TRUE,
    SIGMA_TRUE,
    color="#E45756",
    edgecolor="black",
    marker="*",
    s=220,
    label="Synthetic truth",
    depthshade=False,
    zorder=5,
)
prior_axis.set_xlim(prior_joint_axis_limits["intercept"])
prior_axis.set_ylim(prior_joint_axis_limits["slope"])
prior_axis.set_zlim(prior_joint_axis_limits["sigma"])
prior_axis.set_title("1. Joint prior draws\nFull prior scale")
prior_axis.legend(loc="upper left", fontsize=8)

likelihood_axis.scatter(
    intercept_volume[visible][render_order],
    slope_volume[visible][render_order],
    sigma_volume[visible][render_order],
    c=point_colors[render_order],
    s=11,
    edgecolors="none",
    depthshade=False,
    zorder=1,
)
likelihood_axis.scatter(
    likelihood_intercept_mle,
    likelihood_slope_mle,
    likelihood_sigma_mle,
    color="black",
    edgecolor="white",
    marker="X",
    s=170,
    label="Gaussian MLE",
    depthshade=False,
    zorder=5,
)
likelihood_axis.scatter(
    INTERCEPT_TRUE,
    SLOPE_TRUE,
    SIGMA_TRUE,
    color="#E45756",
    edgecolor="black",
    marker="*",
    s=220,
    label="Synthetic truth",
    depthshade=False,
    zorder=5,
)
likelihood_axis.set_title("2. Likelihood heatmap\nFixed observed x and y")
likelihood_axis.legend(loc="upper left", fontsize=8)

posterior_axis.scatter(
    posterior_intercept_draws[posterior_comparison_order],
    posterior_slope_draws[posterior_comparison_order],
    posterior_sigma_draws[posterior_comparison_order],
    c=posterior_comparison_colors[posterior_comparison_order],
    s=13,
    edgecolors="none",
    depthshade=False,
    zorder=1,
)
posterior_means_for_comparison = joint_draw_matrix.mean(axis=0)
posterior_axis.scatter(
    *posterior_means_for_comparison,
    color="black",
    edgecolor="white",
    marker="D",
    s=130,
    label="Posterior mean",
    depthshade=False,
    zorder=5,
)
posterior_axis.scatter(
    INTERCEPT_TRUE,
    SLOPE_TRUE,
    SIGMA_TRUE,
    color="#E45756",
    edgecolor="black",
    marker="*",
    s=220,
    label="Synthetic truth",
    depthshade=False,
    zorder=5,
)
posterior_axis.set_title("3. Joint posterior draws\nSame scale as likelihood")
posterior_axis.legend(loc="upper left", fontsize=8)

for axis in (prior_axis, likelihood_axis, posterior_axis):
    axis.set(
        xlabel=r"Intercept $\alpha$",
        ylabel=r"Slope $\beta$",
        zlabel=r"Residual scale $\sigma$",
    )
    axis.view_init(elev=24, azim=-58)
    axis.set_box_aspect((1.1, 1.1, 0.9))
for axis in (likelihood_axis, posterior_axis):
    axis.set_xlim(comparison_axis_limits["intercept"])
    axis.set_ylim(comparison_axis_limits["slope"])
    axis.set_zlim(comparison_axis_limits["sigma"])

for axis, label in (
    (prior_axis, r"Relative prior density $p(\theta)/p_{\max}$"),
    (likelihood_axis, r"Relative likelihood $L(\theta)/L_{\max}$"),
    (
        posterior_axis,
        r"Relative posterior kernel $p(\theta)L(\theta)/\max[pL]$",
    ),
):
    panel_mappable = plt.cm.ScalarMappable(cmap=comparison_colormap)
    panel_mappable.set_clim(0, 1)
    panel_mappable.set_array(np.array([0.0, 1.0]))
    panel_colorbar = fig.colorbar(panel_mappable, ax=axis, shrink=0.58, pad=0.08)
    panel_colorbar.set_label(label, fontsize=9)

fig.suptitle(
    r"Joint Bayesian update: $p(\theta\mid\mathbf{x},\mathbf{y})"
    r" \propto p(\theta) L(\theta;\mathbf{x},\mathbf{y})$",
    fontsize=16,
)
plt.show()

joint_bayes_comparison_summary = {
    "prior_draws_visualized": int(prior_parameter_draws.shape[0]),
    "prior_draws_inside_zoom_window": int(prior_inside_comparison_window.sum()),
    "visible_likelihood_grid_points": int(visible.sum()),
    "posterior_draws_visualized": int(joint_draw_matrix.shape[0]),
    "posterior_draws_inside_shared_window": int(posterior_inside_comparison_window.sum()),
    "posterior_draws_outside_shared_window": int((~posterior_inside_comparison_window).sum()),
    "posterior_fraction_inside_shared_window": float(posterior_inside_comparison_window.mean()),
    "likelihood_and_posterior_share_axis_limits": True,
    "comparison_axis_limits": comparison_axis_limits,
    "color_normalization": "separate relative scale in each panel",
}
joint_bayes_comparison_summary

## Posterior predictive check and constructed-data recovery

*Workflow: **Posterior predictive check** within **Evaluate and use model**.*

Posterior means and equal-tail 94% intervals are compared with OLS and the known synthetic truth. Agreement is a debugging observation for one constructed dataset—not posterior calibration or proof of correctness.

The plot asks whether observed responses occupy plausible regions under

$$
p(y^{\mathrm{rep}}\mid x,y)=\int p(y^{\mathrm{rep}}\mid x,\theta)p(\theta\mid x,y)\,d\theta.
$$

It separates:

- posterior uncertainty about $\mu(x)=\alpha+\beta x$;
- wider predictive uncertainty for $y^{\mathrm{rep}}$, including residual variation $\sigma$;
- observed $y$, compared with the predictive band.

This is a visual in-sample check, not an automatic acceptance rule.

In [ ]:
posterior_dataset = posterior["posterior"].dataset
truth_by_parameter = {
    "intercept": INTERCEPT_TRUE,
    "slope": SLOPE_TRUE,
    "sigma": SIGMA_TRUE,
}
ols_by_parameter = {
    "intercept": ols_summary["intercept"],
    "slope": ols_summary["slope"],
    "sigma": ols_summary["residual_sd"],
}
posterior_parameter_summary = {}
for parameter in PARAMETER_NAMES:
    samples = posterior_dataset[parameter]
    lower, upper = samples.quantile(
        [INTERVAL_LOWER_QUANTILE, INTERVAL_UPPER_QUANTILE]
    ).values.tolist()
    truth = truth_by_parameter[parameter]
    posterior_parameter_summary[parameter] = {
        "posterior_mean": float(samples.mean().item()),
        "eti94": [float(lower), float(upper)],
        "mcse_mean": diagnostics_by_parameter[parameter]["mcse_mean"],
        "synthetic_truth": truth,
        "truth_inside_eti94": bool(lower <= truth <= upper),
        "ols_sanity_estimate": ols_by_parameter[parameter],
    }

mu_samples = posterior_dataset["mu"]
predictive_samples = posterior["posterior_predictive"].dataset["y"]
mu_quantiles = mu_samples.quantile(interval_quantiles, dim=("chain", "draw"))
predictive_quantiles = predictive_samples.quantile(interval_quantiles, dim=("chain", "draw"))
fig, axis = plt.subplots(figsize=(9, 5), layout="constrained")
axis.fill_between(
    predictor,
    predictive_quantiles.sel(quantile=INTERVAL_LOWER_QUANTILE),
    predictive_quantiles.sel(quantile=INTERVAL_UPPER_QUANTILE),
    color="#72B7B2",
    alpha=0.22,
    label=r"Pointwise 94% posterior predictive interval for $y^{\mathrm{rep}}$",
)
axis.fill_between(
    predictor,
    mu_quantiles.sel(quantile=INTERVAL_LOWER_QUANTILE),
    mu_quantiles.sel(quantile=INTERVAL_UPPER_QUANTILE),
    color="#4C78A8",
    alpha=0.3,
    label=r"Pointwise 94% posterior interval for mean $\mu(x)$",
)
axis.plot(
    predictor,
    mu_quantiles.sel(quantile=0.5),
    color="#1F4E79",
    label=r"Posterior median of the conditional mean $\mu(x)$",
)
axis.plot(
    predictor,
    true_mean,
    color="#E45756",
    linestyle="--",
    label="Synthetic true mean",
)
axis.scatter(
    predictor,
    observations,
    color="black",
    s=18,
    alpha=0.55,
    label=r"Observed data $y$",
)
axis.set(
    xlabel="Centered reference input x",
    ylabel="Continuous response y",
    title="Posterior predictive check at the fixed observed inputs",
)
axis.legend(loc="upper left")
plt.show()
posterior_parameter_summary

## Scientific review and persistence

*Workflow: provisional review states route the next action.* Diagnostics and predictive checks inform this human judgment; they do not decide automatically.

1. **Computation not valid:** address the implementation or geometry, then refit.
2. **Model not trustworthy:** modify the mean, likelihood, observation process, or priors.
3. **Question unsupported:** stop or narrow the estimand or prediction target.
4. **Provisionally adequate here:** persist the bounded evidence and limitations.

For this constructed-data run, the recorded decision is **adequate for this demonstration** only if execution reproduces the model, stable computation, synthetic recovery, and displayed predictive behavior. It does not authorize causal claims, extrapolation, or transfer to a different data process.

In [ ]:
review_status = "adequate_for_demonstration"
review_rationale = (
    "The fixed constructed-data run showed a finite implementation, stable sampled chains, "
    "descriptive recovery consistent with the known generator and least-squares sanity check, "
    "and an inspectable posterior predictive interval. This decision "
    "is restricted to the synthetic design and inspected features."
)
summary = {
    "schema_version": "bayesian-linear-regression-workflow-summary-v1",
    "question": (
        "How does a continuous response change with a centered reference input, and what "
        "responses should the fitted model predict across the observed design?"
    ),
    "observation": "One fixed reference input and continuous response pair.",
    "population": (
        "Comparable conditionally independent measurements over the declared input range "
        "under a linear mean, constant residual scale, and Gaussian errors."
    ),
    "estimands": {
        "primary": "slope: expected response change per one-unit input increase",
        "secondary": ["intercept at x=0", "residual standard deviation sigma"],
    },
    "model": {
        "mean": "mu_i = intercept + slope * x_i",
        "likelihood": "y_i | mu_i, sigma ~ Normal(mu_i, sigma)",
        "priors": {
            "intercept": (f"Normal({PRIOR_INTERCEPT_MEAN}, {PRIOR_INTERCEPT_SD})"),
            "slope": f"Normal({PRIOR_SLOPE_MEAN}, {PRIOR_SLOPE_SD})",
            "sigma": f"HalfNormal({PRIOR_SIGMA_SD})",
        },
    },
    "settings": {
        "seed": SEED,
        "n_observations": N_OBSERVATIONS,
        "predictor_range": [PREDICTOR_MIN, PREDICTOR_MAX],
        "true_values": truth_by_parameter,
        "prior_predictive_draws": PRIOR_PREDICTIVE_DRAWS,
        "chains": CHAINS,
        "draws": DRAWS,
        "tune": TUNE,
        "cores": CORES,
        "chain_seeds": CHAIN_SEEDS,
        "prior_predictive_seed": SEED + 1,
        "posterior_predictive_seed": SEED + 20,
    },
    "data": data_summary,
    "implementation_checks": implementation_checks,
    "prior_predictive": prior_predictive_summary,
    "least_squares_sanity_check": ols_summary,
    "posterior_parameters": posterior_parameter_summary,
    "posterior_joint_geometry": joint_geometry_summary,
    "joint_bayes_update_comparison": joint_bayes_comparison_summary,
    "diagnostics_by_parameter": diagnostics_by_parameter,
    "diagnostics_overview": diagnostics_overview,
    "package_versions": {"python": platform.python_version(), **package_versions},
    "review": {"status": review_status, "rationale": review_rationale},
    "limitations": [
        "The data were generated from the fitted model family; this is synthetic "
        "self-consistency, not real-world validation.",
        "The predictor is fixed and measured without error; selection and design "
        "uncertainty are outside the model.",
        "The conditional mean is linear and the residual scale is constant with "
        "Gaussian independent errors.",
        "No causal effect is identified, and predictions outside the observed input "
        "range are unsupported.",
        "Least-squares agreement and one-dataset truth recovery are sanity checks, "
        "not posterior calibration.",
        "Favorable diagnostics and one visual posterior predictive check do not prove "
        "general model validity.",
    ],
}

output_directory.mkdir(parents=True, exist_ok=True)
posterior_path = output_directory / "posterior.nc"
summary_path = output_directory / "summary.json"
posterior.to_netcdf(posterior_path, engine="h5netcdf")
summary_path.write_text(
    json.dumps(summary, indent=2, sort_keys=True, allow_nan=False) + "\n",
    encoding="utf-8",
)
with xr.open_datatree(posterior_path, engine="h5netcdf") as restored:
    np.testing.assert_allclose(
        restored["posterior"].dataset["slope"].values,
        posterior["posterior"].dataset["slope"].values,
    )
    np.testing.assert_array_equal(
        restored["posterior_predictive"].dataset["y"].values,
        posterior["posterior_predictive"].dataset["y"].values,
    )
    persisted_groups = restored.groups
{
    "artifacts": {"posterior": posterior_path, "summary": summary_path},
    "round_trip_groups": persisted_groups,
    "review": summary["review"],
}

## Conclusion

The notebook connects a question and estimands to a generative model, prior implications, numerical inference, computational validation, posterior understanding, a posterior predictive check, human review, and reproducible persistence.

A favorable run supports only reproducibility and synthetic self-consistency in the inspected dimensions. It does not establish real-world adequacy, causality, or transportability; those require domain priors, measurement/design review, targeted predictive and sensitivity checks, and a decision context.